# TopicBank: Bank Creation Experiment

Here we are going to collect interpretable topics (automatically, using topic coherence) from multiple model training.
These topics constitute *topic bank*.
And then the topic bank is going to be used for estimating topic models quality in the notebook [TopicBank-Experiment: Model Validation](TopicBank-Experiment-ModelValidation.ipynb).

The process is repeated for several datasets (some of them are already downloadable using [TopicNet](https://github.com/machine-intelligence-laboratory/TopicNet) library).

# Contents<a id="contents"></a>

* [Data](#data)
    * [Coocs](#coocs)
        * [Lower Memory Consumption (or a Bit of Shamanism. Part 1)](#optimizing-memory)
    * [Documents for Coherence Scores](#docs-for-cohs)
        * [Lower Time Consumption in Case of Big Datasets (or a Bit of Shamanism. Part 2)](#optimizing-time)
* [Experiment](#experiment)
    * [Scores](#scores)
    * [Bank Creation](#bank-creation)
* [Postprocessing](#postprocessing)

In [1]:
# General imports

import dill
import itertools
import json
import numpy as np
import os
import pandas as pd
import sys

from enum import Enum
from scipy.stats import gaussian_kde
from matplotlib import pyplot as plt
from tqdm import tqdm
from typing import (
    Dict,
    Iterable,
)

%matplotlib inline

In [2]:
# Making `topnum` module visible for Python

sys.path.insert(0, '../OptimalNumberOfTopics')

In [3]:
# Optimal number of topics

from topicnet.cooking_machine import Dataset

from topnum.data.vowpal_wabbit_text_collection import VowpalWabbitTextCollection
from topnum.scores import (
    PerplexityScore,
    SparsityPhiScore,
    SparsityThetaScore,
)
from topnum.scores.diversity_score import DiversityScore, KNOWN_METRICS
from topnum.scores._base_coherence_score import (
    SpecificityEstimationMethod,
    TextType,
    WordTopicRelatednessType,
)
from topnum.scores.intratext_coherence_score import ComputationMethod
from topnum.search_methods import TopicBankMethod
from topnum.search_methods.topic_bank.topic_bank import TopicBank
from topnum.search_methods.topic_bank.one_model_train_funcs import (
    default_train_func,

    # Functions below are not used (but could have been)

#     regularization_train_func,
#     specific_initial_phi_train_func,
#     background_topics_train_func,

)

## Data<a id="data"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Loading data from disk, creating batches, dictionary, gathering cooccurrence statistics...

In [4]:
DATA_FOLDER_PATH = '/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/dataset_manager'

In [5]:
sorted(os.listdir(DATA_FOLDER_PATH))

['20NG.csv',
 '20NG__internals',
 'Brown',
 'Brown_BOW.csv',
 'Brown_NOOW.csv',
 'MKB10.csv',
 'MKB10__internals',
 'RTL_Wiki.csv',
 'RTL_Wiki_person.csv',
 'RTL_Wiki_person__internals',
 'Reuters',
 'Reuters_BOW.csv',
 'Reuters_NOOW.csv',
 'WikiRef-220',
 '__init__.py',
 '__pycache__',
 'api.py',
 'postnauka.csv',
 'postnauka__internals',
 'ruwiki_good.txt',
 'ruwiki_good__internals',
 'wiki_ref220_bow.csv',
 'wiki_ref220_natural_order.csv']

In [6]:
class DatasetName(Enum):
    POSTNAUKA = 'Post_Science'
    # REUTERS = 'Reuters'
    # BROWN = 'Brown'
    TWENTY_NEWSGROUPS = '20_Newsgroups'
    GOOD_RU_WIKI = 'Good_RU_Wiki'

In [7]:
DATASET_NAME_TO_DATASET_FILE_PATH = {
    DatasetName.POSTNAUKA: os.path.join(
        DATA_FOLDER_PATH, 'postnauka.csv'
    ),
    # DatasetName.REUTERS: os.path.join(
    #     DATA_FOLDER_PATH, 'Reuters.csv'
    # ),
    # DatasetName.BROWN: os.path.join(
    #     DATA_FOLDER_PATH, 'Brown.csv'
    # ),
    DatasetName.TWENTY_NEWSGROUPS: os.path.join(
        DATA_FOLDER_PATH, '20NG.csv'
    ),
    # DatasetName.AG_NEWS: os.path.join(
    #     DATA_FOLDER_PATH, 'AG_News.csv'
    # ),
    # DatasetName.WATAN: os.path.join(
    #     DATA_FOLDER_PATH, 'Watan2004.csv'
    # ),
    # DatasetName.HABRAHABR: os.path.join(
    #     DATA_FOLDER_PATH, 'Habrahabr.csv'
    # ),
    DatasetName.GOOD_RU_WIKI: os.path.join(
        DATA_FOLDER_PATH, 'ruwiki_good.txt'
    ),
}

In [8]:
DATASET_NAME = DatasetName.GOOD_RU_WIKI  # select a dataset here

DATASET_FILE_PATH = DATASET_NAME_TO_DATASET_FILE_PATH[DATASET_NAME]

Checking if all OK with data, what modalities does the collection have.

In [9]:
! head -n 2 $DATASET_FILE_PATH

Санкт-Петербург |@lemmatized год:301 петроград:7 ленинград:18 численность:14 население:33 город:212 россия:32 .:677 федеральный:15 значение:8 административный:5 центр:29 округа:3 ленинградский:16 область:6 основать:5 царь:3 <person>:195 являться:34 столица:19 российский:35 государство:9 назвать:4 честь:6 святой:11 небесный:2 покровитель:2 основатель:3 время:17 стать:25 большой:23 ассоциироваться:1 имя:34 исторически:1 культурно:1 связать:2 рождение:1 империя:7 вхождение:1 современный:7 история:11 роль:5 европейский:3 великий:5 держава:1 расположить:11 страна:16 побережье:4 финский:16 залив:15 устье:4 река:21 нева:31 находиться:17 конституционный:1 суд:5 федерация:14 геральдический:2 совет:8 президент:1 орган:6 власть:11 межпарламентский:3 ассамблея:3 снг:2 разместить:1 главный:9 командование:2 флот:1 штаб:2 западный:8 военный:8 вооружённый:3 сила:6 быть:74 революция:6 февральский:2 октябрьский:6 ход:4 отечественный:4 война:10 блокада:7 результат:12 миллион:37 человек:39 погибнуть:4 объ

In [10]:
def get_dataset_internals_folder_path(dataset_name: DatasetName) -> str:
    return os.path.join('.', dataset_name.value + '__internals')

In [11]:
DATASET_INTERNALS_FOLDER_PATH = get_dataset_internals_folder_path(DATASET_NAME)

In [12]:
DATASET_INTERNALS_FOLDER_PATH

'./Good_RU_Wiki__internals'

In [13]:
%%time

# If using really big datasets (like Habrahabr),
# one may need to set this equal `False`
KEEP_DATASET_IN_MEMORY = True

DATASET = Dataset(
    DATASET_FILE_PATH,
    internals_folder_path=DATASET_INTERNALS_FOLDER_PATH,
    keep_in_memory=KEEP_DATASET_IN_MEMORY,
)

CPU times: user 5.67 s, sys: 722 ms, total: 6.39 s
Wall time: 6.31 s


Looking what is inside dataset's folder

In [14]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt',
 'result_50',
 'result2_50',
 'result2',
 'batches',
 'dict.dict',
 'result']

Creating batches

In [15]:
DATASET.get_batch_vectorizer()

artm.BatchVectorizer(data_path="./Good_RU_Wiki__internals/batches", num_batches=9)

In [16]:
os.listdir(DATASET_INTERNALS_FOLDER_PATH)

['vw.txt',
 'result_50',
 'result2_50',
 'result2',
 'batches',
 'dict.dict',
 'result']

In [17]:
if KEEP_DATASET_IN_MEMORY:
    DOCUMENTS = list(DATASET._data.index)
else:
    DOCUMENTS = list(DATASET._data_index)

NUM_DOCUMENTS = len(DOCUMENTS)

print(f'Num documents: {NUM_DOCUMENTS}')

Num documents: 8603


Let's look at some text samples

In [18]:
DATASET._data.head()

,vw_text,raw_text,id
id,,,
Санкт-Петербург,Санкт-Петербург |@lemmatized год:301 петроград...,,Санкт-Петербург
Дворцовая_площадь,Дворцовая_площадь |@lemmatized дворцовый:43 пл...,,Дворцовая_площадь
Греко-персидские_войны,Греко-персидские_войны |@lemmatized грёкий:23 ...,,Греко-персидские_войны
Тихий_океан,Тихий_океан |@lemmatized тихий:92 океан:174 ус...,,Тихий_океан
Атлантический_океан,Атлантический_океан |@lemmatized атлантический...,,Атлантический_океан


In [19]:
DATASET.get_possible_modalities()

{'@categories', '@lemmatized', '@ngramms'}

In [20]:
MAIN_MODALITY = '@lemmatized'

In [21]:
DATASET.get_dictionary()

artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=892938)

In [22]:
dictionary = DATASET.get_dictionary()

In [23]:
print(dictionary)

for modality in DATASET.get_possible_modalities():
    if modality not in [MAIN_MODALITY]:
        dictionary.filter(class_id=modality, max_df=0, inplace=True)

artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=892938)


In [24]:
dictionary.filter(min_df=5, max_df_rate=0.5)

artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=61688)

In [25]:
DATASET._cached_dict = dictionary

In [26]:
DATASET.get_dictionary()

artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=61688)

In [27]:
import scipy

from typing import List

from topicnet.cooking_machine.models.base_regularizer import BaseRegularizer
from topicnet.cooking_machine.models.thetaless_regularizer import (
    dataset2sparse_matrix,
)
from topicnet.cooking_machine.models import (
    BaseScore as BaseTopicNetScore,
    TopicModel
)

In [28]:
def calc_doc_occurrences(dataset, modality):
    """
    :param n_dw_matrix: sparse document-word matrix, shape is D x W
    :return: sparse matrix of co-occurrences

    doc_occurrences[w1, w2] = the number of the documents
    where there are w1 and w2
    """
    n_dw_matrix = dataset2sparse_matrix(dataset, modality, modalities_to_use=[modality])
    matrix = (scipy.sparse.csc_matrix(n_dw_matrix) > 0).astype(int)
    co_occurrences = matrix.T * matrix

    return co_occurrences.diagonal(), co_occurrences


def create_pmi_top_function(
    doc_occurrences, doc_co_occurrences,
    documents_number, top_sizes,
    topic_indices=None,
    co_occurrences_smooth=1.
):
    """
    :param doc_occurrences: array of doc occurrences of words
    :param doc_co_occurrences: sparse matrix of doc co-occurrences of words
    :param documents_number: number of the documents
    :param top_sizes: list of top values to calculate top-pmi for
    :param co_occurrences_smooth: constant to smooth co-occurrences in log
    :return: function which takes phi and theta and returns
    pair of two arrays: pmi-s of the tops and ppmi-s of the tops

    pmi[i] - pmi(top of size top_sizes[i])
    ppmi[i] - ppmi(top of size top_sizes[i])

    pmi(words) = sum_{u in words, v in words, u != v}
    log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    )

    ppmi(words) = sum_{u in words, v in words, u != v}
    max(log(
        (doc_co_occurrences[u, v] * documents_number + co_occurrences_smooth)
        / doc_occurrences[u] / doc_occurrences[v]
    ), 0)

    """
    def func(phi):
        T, W = phi.shape
        # T = len(topic_indices)
        topic_indices = list(range(T))

        max_top_size = max(top_sizes)
        topic_pmis, topic_ppmis = dict(), dict()
        pmi, ppmi = np.zeros(max_top_size), np.zeros(max_top_size)
        tops = np.argpartition(phi, -max_top_size, axis=1)[:, -max_top_size:]
        
        for t in topic_indices:
            top = sorted(tops[t], key=lambda w: - phi[t, w])
            # print(top, phi.shape, doc_co_occurrences.shape)
            co_occurrences = doc_co_occurrences[top, :][:, top].todense()
            occurrences = doc_occurrences[top]
            values = np.log(
                (co_occurrences * documents_number + co_occurrences_smooth)
                / (occurrences[:, np.newaxis] * occurrences[np.newaxis, :] + co_occurrences_smooth)
            )
            diag = np.diag_indices(len(values))
            # values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()

            current_pmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_pmis[t] = current_pmi
            pmi += current_pmi

            values[values < 0.] = 0.
            current_ppmi = np.array(
               values.cumsum(axis=0).cumsum(axis=1)[diag] - values[diag].cumsum()
            ).ravel()
            topic_ppmis[t] = current_ppmi
            ppmi += current_ppmi
            
        sizes = np.arange(2, max_top_size + 1)
        pmi[1:] /= (T * sizes * (sizes - 1))
        ppmi[1:] /= (T * sizes * (sizes - 1))
        indices = np.array(top_sizes) - 1

        for t in topic_indices:
            topic_pmis[t][1:] /= (sizes * (sizes - 1))
            topic_ppmis[t][1:] /= (sizes * (sizes - 1))

        result_topic_pmis = {t: p[indices] for t, p in topic_pmis.items()}
        result_topic_ppmis = {t: p[indices] for t, p in topic_ppmis.items()}

        return pmi[indices], ppmi[indices], result_topic_pmis, result_topic_ppmis

    return func

In [29]:
%%time

occurences, co_occurences = calc_doc_occurrences(DATASET, MAIN_MODALITY)

CPU times: user 47.1 s, sys: 1.3 s, total: 48.4 s
Wall time: 47.9 s


In [30]:
co_occurences.shape

(61688, 61688)

In [31]:
calc_pmi = create_pmi_top_function(
    occurences, co_occurences,
    DATASET.get_dataset().shape[0], [20],
    topic_indices=[0, 1, 2],
    co_occurrences_smooth=1e-2,
)

In [32]:
import copy


class TopTokenCoherence(BaseTopicNetScore):
    def __init__(self, name, func):
        super().__init__()

        self._name = name
        self.calc_pmi = func

    @property
    def name(self):
        return self._name

    def call(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[1]

    def call_by_topic(self, model: TopicModel):
        values = self.calc_pmi(model.get_phi_dense()[0].T)

        return values[3]

    def compute(
            self,
            model,
            topics: List[str] = None,
            documents: List[str] = None) -> Dict[str, float]:

        values = self.call_by_topic(model)

        phi = model.get_phi()

        if topics is None:
            topics = list(phi.columns)

            if hasattr(model, 'has_bcg'):
                print(f'Detected bcg topics! Skipping for coherence computation (and will have {len(topics) - 1} topics).')

                topics = topics[:-1]
        else:
            assert False

        index2topic = {phi.columns.get_loc(t): t for t in topics}
        topic2index = {t: i for i, t in index2topic.items()}

        if hasattr(model, 'has_bcg'):
            assert list(index2topic.keys()) == list(values.keys())[:-1]
        else:
            assert list(index2topic.keys()) == list(values.keys())

        result = {
            t: float(values[topic2index[t]])
            for t in topics
        }

        assert len(result) == len(index2topic)

        return result

    def _attach(self, model: TopicModel):
        if self._name in model.custom_scores:
            print(
                f'Score with such name "{self._name}" already attached to model!'
                f' So rewriting it...'
                f' All model\'s custom scores: {list(model.custom_scores.keys())}'
            )

        # TODO: TopicModel should provide ability to add custom scores
        model.custom_scores[self.name] = copy.deepcopy(self)

## Experiment<a id="experiment"></a>

Finally we are getting to the main part!)

### Scores (for Topics and Models)<a id="scores"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we define a lot of scores (which mainly differ in initial parameters).

In [33]:
ONE_MODEL_NUM_TOPICS = 50
NUM_TOP_WORDS = 20

In [34]:
top = NUM_TOP_WORDS
target_topic_indices = list(range(ONE_MODEL_NUM_TOPICS))

coherence_score = TopTokenCoherence(
    name=f'coherence_{top}',
    func=create_pmi_top_function(
        occurences, co_occurences,
        DATASET.get_dataset().shape[0], [top],
        # topic_indices=target_topic_indices,
        co_occurrences_smooth=1e-2,
    )
)

diversity_scores = [
    DiversityScore(
        name=f'diversity_{metric}',
        metric=metric,
        class_ids=MAIN_MODALITY,
    )

    for metric in KNOWN_METRICS
]

Other coherence score variations

And a pair of default ARTM scores (these ones are fast)

In [35]:
other_scores = [
    PerplexityScore(
        name='perplexity'
    ),
]

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [36]:
NUM_ITERATIONS = 10

In [37]:
seed = 0

In [38]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = default_train_func  # default train func

In [39]:
DATASET_INTERNALS_FOLDER_PATH

'./Good_RU_Wiki__internals'

In [40]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result_50'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [41]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls -alh $DATASET_INTERNALS_FOLDER_PATH

./Good_RU_Wiki__internals
total 248M
drwxrwxr-x  7 alekseev_v mil_lab 4,0K мар 27 18:18 .
drwxrwxr-x 11 alekseev_v mil_lab 4,0K мар 27 18:18 ..
drwxrwxr-x  2 alekseev_v mil_lab 4,0K мар 24 10:57 batches
-rw-rw-r--  1 alekseev_v mil_lab  49M мар 24 10:57 dict.dict
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 24 11:04 result
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 24 12:21 result2
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 27 16:00 result2_50
drwxrwxr-x  3 alekseev_v mil_lab 4,0K мар 27 17:47 result_50
-rw-rw-r--  1 alekseev_v mil_lab 200M мар 24 10:57 vw.txt


In [42]:
SEARCH_RESULTS_FOLDER_PATH

'./Good_RU_Wiki__internals/result_50'

In [45]:
! ls $SEARCH_RESULTS_FOLDER_PATH

ls: cannot access './Good_RU_Wiki__internals/result_50': No such file or directory


In [46]:
BANK_FOLDER_PATH

'./Good_RU_Wiki__internals/result_50/bank__0'

In [47]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [48]:
seed

0

In [49]:
ONE_MODEL_NUM_TOPICS

50

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [50]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 90,

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

In [51]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [52]:
! echo $DATASET_INTERNALS_FOLDER_PATH
! ls $DATASET_INTERNALS_FOLDER_PATH

./Good_RU_Wiki__internals
batches  dict.dict  result  result2  result2_50  result_50  vw.txt


In [53]:
optimizer._save_file_path

'./Good_RU_Wiki__internals/result_50/search_result__0.json'

In [54]:
optimizer._topic_bank._path

'./Good_RU_Wiki__internals/result_50/bank__0'

Fulfilling the search (get ready for a really long process!):

In [55]:
%%time

optimizer.search_for_optimum(DATASET)

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.50it/s]


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 23701.962890625, 'coherence_20': 1.271056255022922, 'diversity_euclidean': 0.06817160745189306, 'diversity_jensenshannon': 0.684349668889001, 'diversity_hellinger': 0.7904835425832982, 'diversity_cosine': 0.8860105268866816, 'perplexity': 23701.962890625, 'ppl_fair': 23701.962890625, 'ppl_cheatty': 8288.0302734375}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.25it/s]
Creating first level with 5 topics. Dictionary: artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=61688).
Copying phi for the first level. Phi shape: (61688, 5). First words: MultiIndex([('@lemmatized',              'рлэ'),
            ('@lemmatized',      'локационный'),
            ('@lemmatized', 'неподверженность'),
            ('@lemmatized',         'даякский'),
            ('@lemmatized',           'рэдкот'),
            ('@lemmatized',        'галоархея'),
            ('@lemmatized',          'babcock'),
            ('@lemmatized

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 23701.962890625, 'coherence_20': 1.271056255022922, 'diversity_euclidean': 0.06817160745187226, 'diversity_jensenshannon': 0.6843496688865095, 'diversity_hellinger': 0.790483542581285, 'diversity_cosine': 0.8860105268855312, 'perplexity': 23701.962890625, 'ppl_fair': 23701.962890625, 'ppl_cheatty': 8288.0302734375}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.44it/s]
Creating first level with 5 topics. Dictionary: artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=61688).
Copying phi for the first level. Phi shape: (61688, 5). First words: MultiIndex([('@lemmatized',              'рлэ'),
            ('@lemmatized',      'локационный'),
            ('@lemmatized', 'неподверженность'),
            ('@lemmatized',         'даякский'),
            ('@lemmatized',           'рэдкот'),
            ('@lemmatized',        'галоархея'),
            ('@lemmatized',          'babcock'),
            ('@lemmatized

...

Bank scores: {'perplexity_score': 12935.0205078125, 'coherence_20': 1.325640369792121, 'diversity_euclidean': 0.06571734438226805, 'diversity_jensenshannon': 0.6800217140392609, 'diversity_hellinger': 0.7841219651265123, 'diversity_cosine': 0.8840728042243101, 'perplexity': 12935.0205078125, 'ppl_fair': 12935.0205078125, 'ppl_cheatty': 7393.04638671875}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.42it/s]
Creating first level with 10 topics. Dictionary: artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=61688).
Copying phi for the first level. Phi shape: (61688, 10). First words: MultiIndex([('@lemmatized',              'рлэ'),
            ('@lemmatized',      'локационный'),
            ('@lemmatized', 'неподверженность'),
            ('@lemmatized',         'даякский'),
            ('@lemmatized',           'рэдкот'),
            ('@lemmatized',        'галоархея'),
            ('@lemmatized',          'babcock'),
            ('@lem

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 12935.0205078125, 'coherence_20': 1.325640369792121, 'diversity_euclidean': 0.06571734438226257, 'diversity_jensenshannon': 0.6800217140394483, 'diversity_hellinger': 0.7841219651265428, 'diversity_cosine': 0.8840728042250905, 'perplexity': 12935.0205078125, 'ppl_fair': 12935.0205078125, 'ppl_cheatty': 7393.04638671875}
100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.51it/s]
Creating first level with 10 topics. Dictionary: artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=61688).
Copying phi for the first level. Phi shape: (61688, 10). First words: MultiIndex([('@lemmatized',              'рлэ'),
            ('@lemmatized',      'локационный'),
            ('@lemmatized', 'неподверженность'),
            ('@lemmatized',         'даякский'),
            ('@lemmatized',           'рэдкот'),
            ('@lemmatized',        'галоархея'),
            ('@lemmatized',          'babcock'),
            ('@lem

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 12935.0205078125, 'coherence_20': 1.325640369792121, 'diversity_euclidean': 0.0657173443822478, 'diversity_jensenshannon': 0.6800217140392175, 'diversity_hellinger': 0.7841219651260267, 'diversity_cosine': 0.884072804224423, 'perplexity': 12935.0205078125, 'ppl_fair': 12935.0205078125, 'ppl_cheatty': 7393.0458984375}
100%|████████████████████████████████████████████| 20/20 [1:21:58<00:00, 245.95s/it]
CPU times: user 2h 26min 43s, sys: 3min 38s, total: 2h 30min 21s
Wall time: 1h 21min 59s


What topics we have in bank

In [57]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1   topic_2       topic_3  \
@lemmatized рлэ               0.000000e+00      0.0  0.000000  0.000000e+00   
            локационный       1.026262e-15      0.0  0.000005  0.000000e+00   
            неподверженность  0.000000e+00      0.0  0.000020  0.000000e+00   
            даякский          1.499791e-10      0.0  0.000000  1.916781e-16   
            рэдкот            0.000000e+00      0.0  0.000000  0.000000e+00   

                              topic_4       topic_5   topic_6       topic_7  \
@lemmatized рлэ                   0.0  0.000000e+00  0.000000  0.000000e+00   
            локационный           0.0  1.051820e-14  0.000000  1.111689e-15   
            неподверженность      0.0  0.000000e+00  0.000000  0.000000e+00   
            даякский              0.0  0.000000e+00  0.000014  0.000000e+00   
            рэдкот                0.0  0.000000e+00  0.000000  0.000000e+00   

                              topic_8  topic_9  
@lemmatized рлэ                   0.0      0.0  
            локационный           0.0      0.0  
            неподверженность      0.0      0.0  
            даякский              0.0      0.0  
            рэдкот                0.0      0.0

In [58]:
bank_topics = optimizer._topic_bank.view_topics()

In [59]:
bank_topics.shape

(61688, 10)

In [60]:
bank_topics.head()

topic_0  topic_1   topic_2       topic_3  \
@lemmatized рлэ               0.000000e+00      0.0  0.000000  0.000000e+00   
            локационный       1.026262e-15      0.0  0.000005  0.000000e+00   
            неподверженность  0.000000e+00      0.0  0.000020  0.000000e+00   
            даякский          1.499791e-10      0.0  0.000000  1.916781e-16   
            рэдкот            0.000000e+00      0.0  0.000000  0.000000e+00   

                              topic_4       topic_5   topic_6       topic_7  \
@lemmatized рлэ                   0.0  0.000000e+00  0.000000  0.000000e+00   
            локационный           0.0  1.051820e-14  0.000000  1.111689e-15   
            неподверженность      0.0  0.000000e+00  0.000000  0.000000e+00   
            даякский              0.0  0.000000e+00  0.000014  0.000000e+00   
            рэдкот                0.0  0.000000e+00  0.000000  0.000000e+00   

                              topic_8  topic_9  
@lemmatized рлэ                   0.0      0.0  
            локационный           0.0      0.0  
            неподверженность      0.0      0.0  
            даякский              0.0      0.0  
            рэдкот                0.0      0.0

In [63]:
bank_topics['topic_6'].sort_values(ascending=False)[:20]

@lemmatized  сербский       0.018557
             серб           0.013351
             хорватский     0.010837
             сербия         0.008925
             хорватия       0.008571
             территория     0.008101
             гонконг        0.007632
             югославия      0.006734
             болгарский     0.006306
             босния         0.005951
             болгария       0.005526
             страна         0.005063
             блюдо          0.004687
             югославский    0.004676
             население      0.004452
             хорват         0.004188
             республика     0.003726
             кухня          0.003415
             местный        0.003392
             герцеговина    0.003382
Name: topic_6, dtype: float64

And topic scores

In [64]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9
kernel_size,4340.000000,4340.000000,5921.000000,4604.000000,4449.000000,5229.000000,5372.000000,4399.000000,6615.000000,5655.000000
coherence_20,1.504778,1.177245,1.148254,1.300758,1.224246,1.507856,1.413234,1.659403,1.101885,1.218744
distance_to_nearest,0.000000,0.938270,0.911490,0.874375,0.665535,0.847624,0.871184,0.877588,0.865746,0.727888


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [65]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [59]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [60]:
optimizer._result['num_bank_topics']

[5]

In [61]:
len(optimizer._result['bank_topic_scores'])

1

In [66]:
import artm
from topnum.model_constructor import KnownModel, init_plsa
from topicnet.cooking_machine.rel_toolbox_lite import (
    count_vocab_size,
    transform_regularizer,
)
from topicnet.cooking_machine.model_constructor import (
    add_standard_scores,
    create_default_topics,
    init_model,
)

def init_model_from_family(
        family: str or KnownModel,
        dataset: Dataset,
        main_modality: str,
        num_topics: int,
        seed: int,
        specific_topic_names = None,
        modalities_to_use: List[str] = None,
        num_processors: int = 3,
        model_params: dict = None,
):
    """
    Returns
    -------
    model: TopicModel() instance
    """
    if isinstance(family, KnownModel):
        family = family.value

    if modalities_to_use is None:
        modalities_to_use = [main_modality]

    custom_regs = {}

    if family == "LDA":
        model = init_lda(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "PLSA":
        model = init_plsa(
            dataset, modalities_to_use, main_modality, num_topics
        )
    elif family == "TARTM":
        model, custom_regs = init_thetaless(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "sparse":
        model = init_bcg_sparse_model(
            dataset, modalities_to_use, main_modality, num_topics, 1, model_params
        )
    elif family == "decorrelation":
        model = init_decorrelated_plsa(
            dataset, modalities_to_use, main_modality, num_topics, model_params
        )
    elif family == "ARTM":
        model = init_baseline_artm(
            dataset, modalities_to_use, main_modality, num_topics, 1, specific_topic_names, model_params
        )
    else:
        raise ValueError(f'family: {family}')

    model.num_processors = num_processors

    if seed is not None:
        model.seed = seed

    dictionary = dataset.get_dictionary()

    # TODO: maybe this cycle is not necessary
    for modality in dataset.get_possible_modalities():
        if modality not in modalities_to_use:
            dictionary.filter(class_id=modality, max_df=0, inplace=True)

    model.initialize(dictionary)
    add_standard_scores(model, dictionary, main_modality=main_modality,
                        all_modalities=modalities_to_use)

    model = TopicModel(
        artm_model=model,
        custom_regularizers=custom_regs
    )
    model.has_bcg = True  # TODO: only if init_bcg_sparse_model

    return model


def init_bcg_sparse_model(
        dataset,
        modalities_to_use,
        main_modality,
        specific_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str or dict
    main_modality : str
    specific_topics : int
    bcg_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_plsa(
        dataset, modalities_to_use, main_modality, specific_topics, bcg_topics
    )
    background_topic_names = model.topic_names[-bcg_topics:]

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    dictionary = dataset.get_dictionary()
    baseline_class_ids = {class_id: 1 for class_id in modalities_to_use}
    data_stats = count_vocab_size(dictionary, baseline_class_ids)

    # all coefficients are relative
    regularizers = [
        artm.SmoothSparsePhiRegularizer(
             name='smooth_phi_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
             class_ids=[main_modality],
        ),
        artm.SmoothSparseThetaRegularizer(
             name='smooth_theta_bcg',
             topic_names=background_topic_names,
             tau=model_params.get("smooth_bcg_tau", 0.1),
        ),
        artm.SmoothSparsePhiRegularizer(
             name='sparse_phi_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
             class_ids=[main_modality],
            ),
        artm.SmoothSparseThetaRegularizer(
             name='sparse_theta_sp',
             topic_names=specific_topic_names,
             tau=model_params.get("sparse_sp_tau", -0.05),
        ),
    ]
    for reg in regularizers:
        model.regularizers.add(transform_regularizer(
            data_stats,
            reg,
            model.class_ids,
            n_topics=len(reg.topic_names)
        ))

    return model


def init_baseline_artm(
        dataset,
        modalities_to_use,
        main_modality,
        num_topics,
        bcg_topics,
        specific_topic_names = None,
        model_params: dict = None,
):
    """
    Creates simple artm model with standard scores.

    Parameters
    ----------
    dataset : Dataset
    modalities_to_use : list of str
    main_modality : str
    num_topics : int

    Returns
    -------
    model: artm.ARTM() instance
    """
    if model_params is None:
        model_params = dict()

    model = init_bcg_sparse_model(
        dataset, modalities_to_use, main_modality, num_topics, bcg_topics, specific_topic_names, model_params
    )

    if specific_topic_names is None:
        print('No spec topics')
        specific_topic_names = model.topic_names[:-bcg_topics]

    model.regularizers.add(
        artm.DecorrelatorPhiRegularizer(
            gamma=0,
            tau=model_params.get('decorrelation_tau', 0.01),
            name='decorrelation',
            topic_names=specific_topic_names,
            class_ids=modalities_to_use,
        )
    )

    return model

In [67]:
def artm_train_func(
        dataset: Dataset,
        model_number: int,
        num_topics: int,
        num_fit_iterations: int,
        scores: List = None,
        **kwargs) -> TopicModel:
    """

    Additional Parameters
    ---------------------
    kwargs
        Some params for `_get_topic_model`, such as `cache_theta` and `num_processors`
    """

    topic_model = init_model_from_family(
        family='ARTM',
        dataset=DATASET,
        main_modality=MAIN_MODALITY,
        num_topics=ONE_MODEL_NUM_TOPICS,
        seed=model_number,
        model_params={
            'decorrelation_tau': 0.01,  # best values
            'smooth_bcg_tau': 0.05,
            'sparse_sp_tau': -0.05,
        }
    )

    num_fit_iterations_with_scores = 1

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=max(0, num_fit_iterations - num_fit_iterations_with_scores)
    )
    _fit_model_with_scores(
        topic_model,
        DATASET,
        scores,
        num_fit_iterations=num_fit_iterations_with_scores
    )

    return topic_model


def _fit_model_with_scores(
        topic_model: TopicModel,
        dataset: Dataset,
        scores: List = None,
        num_fit_iterations: int = 1):

    if scores is not None:
        for score in scores:
            score._attach(topic_model)

    topic_model._fit(
        dataset.get_batch_vectorizer(),
        num_iterations=num_fit_iterations
    )

### Bank Creation<a id="bank-creation"></a>

<div style="text-align: right">Back to <a href=#contents>Contents</a></div>

Here we finally run the experiment!

In [69]:
ONE_MODEL_NUM_TOPICS

50

In [70]:
NUM_ITERATIONS = 10

In [71]:
seed = 0

In [72]:
# We use only one train function here
# Other variations are also possible
# It would be even better to make bank using several train functions
# However, it would also take way more time 

TRAIN_FUNCS = artm_train_func  # default train func

In [67]:
DATASET_INTERNALS_FOLDER_PATH

'./Good_RU_Wiki__internals'

In [73]:
SEARCH_RESULTS_FOLDER_PATH = os.path.join(
    DATASET_INTERNALS_FOLDER_PATH, 'result2_50'
)

# File with some info about the process
SEARCH_RESULT_FILE_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'search_result__{seed}.json'
)

# Bank, with topics and their score values
BANK_FOLDER_PATH = os.path.join(
    SEARCH_RESULTS_FOLDER_PATH, f'bank__{seed}'
)

In [74]:
SEARCH_RESULTS_FOLDER_PATH

'./Good_RU_Wiki__internals/result2_50'

In [78]:
BANK_FOLDER_PATH

'./Good_RU_Wiki__internals/result2_50/bank__0'

In [76]:
! ls $BANK_FOLDER_PATH

model_0__phi.bin	    model_4__phi.bin
model_0__topic_scores.bin   model_4__topic_scores.bin
model_10__phi.bin	    model_5__phi.bin
model_10__topic_scores.bin  model_5__topic_scores.bin
model_11__phi.bin	    model_6__phi.bin
model_11__topic_scores.bin  model_6__topic_scores.bin
model_12__phi.bin	    model_7__phi.bin
model_12__topic_scores.bin  model_7__topic_scores.bin
model_1__phi.bin	    model_8__phi.bin
model_1__topic_scores.bin   model_8__topic_scores.bin
model_2__phi.bin	    model_9__phi.bin
model_2__topic_scores.bin   model_9__topic_scores.bin
model_3__phi.bin	    topics.bin
model_3__topic_scores.bin   topic_scores.bin


In [79]:
os.makedirs(SEARCH_RESULTS_FOLDER_PATH, exist_ok=True)
os.makedirs(BANK_FOLDER_PATH, exist_ok=True)

In [80]:
seed

0

One cay vary some parameters below (for example `max_num_models` and `num_fit_iterations`).

In [83]:
optimizer = TopicBankMethod(
    data        = DATASET,
    main_modality = MAIN_MODALITY,
    
    min_df_rate = 0.0,  # dictionary filtering has already been done little earlier
    max_df_rate = 1.0,  #   so we don't want these parameters to have any effect

    main_topic_score   = coherence_score,
    other_topic_scores = [],
    other_scores       = [coherence_score] + diversity_scores + other_scores,
   # documents          = TEST_DOCUMENTS,

    start_model_number   = 0,
    max_num_models       = 20,
    one_model_num_topics = ONE_MODEL_NUM_TOPICS,  # 100,
    num_fit_iterations   = NUM_ITERATIONS,  # 100,  # 100 should be enough;
                                 # however, for big data better to reduce this one
                                 # (otherwise the process will be too slow)

    topic_score_threshold_percentile = 0.9918260892662304,  # DIFF ALSO HERE

    save_bank         = True,
    save_model_topics = True,
    save_file_path    = SEARCH_RESULT_FILE_PATH,
    bank_folder_path  = BANK_FOLDER_PATH,

    train_funcs = TRAIN_FUNCS,
    
    verbose = True,
)

# TODO: use Holdout Perplexity as Stop score

/home/alekseev_v/projects/iterative/../OptimalNumberOfTopics/topnum/search_methods/topic_bank/topic_bank_method.py:208: UserWarning: topic_score_threshold_percentile 0.9918260892662304 is less than one! It is expected to be in [0, 100]. Are you sure you want to proceed (yes/no)?
  warnings.warn(


In [84]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

Checking file paths

In [85]:
! ls ./Post_Science__internals

batches    _result  _result2  result2_50  result_unfiltered_dict
dict.dict  result   result2   result_50   vw.txt


In [86]:
optimizer._save_file_path

'./Good_RU_Wiki__internals/result2_50/search_result__0.json'

In [87]:
optimizer._topic_bank._path

'./Good_RU_Wiki__internals/result2_50/bank__0'

Fulfilling the search (get ready for a really long process!):

In [88]:
%%time

optimizer.search_for_optimum(DATASET)

  0%|                                                        | 0/20 [00:00<?, ?it/s]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 50 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.58it/s]
Using absoulte threshold: 0.9918260892662304.
Skipping saving scores for bcg topic


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 15662.7099609375, 'coherence_20': 1.1425703063188861, 'diversity_euclidean': 0.07887198216968297, 'diversity_jensenshannon': 0.6946457152370844, 'diversity_hellinger': 0.806890043129983, 'diversity_cosine': 0.9036317622655906, 'perplexity': 15662.7099609375, 'ppl_fair': 15662.7099609375, 'ppl_cheatty': 7013.5810546875}
  5%|██▎                                          | 1/20 [04:04<1:17:16, 244.00s/it]No spec topics
No spec topics


...

Bank scores: {'perplexity_score': 11298.396484375, 'coherence_20': 1.1589140383647176, 'diversity_euclidean': 0.0691448355362054, 'diversity_jensenshannon': 0.6896389671004165, 'diversity_hellinger': 0.800975990258223, 'diversity_cosine': 0.8826898277031303, 'perplexity': 11298.396484375, 'ppl_fair': 11298.396484375, 'ppl_cheatty': 6092.97412109375}
 95%|█████████████████████████████████████████▊  | 19/20 [1:41:00<05:38, 338.05s/it]No spec topics
No spec topics


/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).
Detected bcg topics! Skipping for diversity computation (and now 50 topics).

  0%|                                                         | 0/1 [00:00<?, ?it/s]Detected bcg topics! Skipping for coherence computation (and will have 50 topics).

100%|█████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.30it/s]
Using absoulte threshold: 0.9918260892662304.
Creating first level with 22 topics. Dictionary: artm.Dictionary(name=c8880f58-b4a8-459d-939b-6d4e17581a43, num_entries=61688).
Copying phi for the first level. Phi shape: (61688, 22). First words: MultiIndex([('@lemmatized',              'рлэ'),
            ('@lemmatized',      'локационный'),
            ('@lemmatized', 'неподверженность'),
            ('@lemmatized',         'даякский')

/home/alekseev_v/miniconda3/envs/topicnet/lib/python3.8/site-packages/topicnet/cooking_machine/model_constructor.py:37: UserWarning: Parameter `dictionary` is obsolete: it is not used in the function "add_standard_scores"!
  warnings.warn(


Bank scores: {'perplexity_score': 11298.396484375, 'coherence_20': 1.1589140383647176, 'diversity_euclidean': 0.06914483553623833, 'diversity_jensenshannon': 0.6896389671000702, 'diversity_hellinger': 0.8009759902579212, 'diversity_cosine': 0.8826898277026235, 'perplexity': 11298.396484375, 'ppl_fair': 11298.396484375, 'ppl_cheatty': 6092.97412109375}
100%|████████████████████████████████████████████| 20/20 [1:46:23<00:00, 319.18s/it]
CPU times: user 3h 11min 58s, sys: 4min 41s, total: 3h 16min 39s
Wall time: 1h 46min 24s


In [89]:
optimizer._main_modality

'@lemmatized'

What topics we have in bank

In [90]:
optimizer._topic_bank.view_topics().head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized рлэ                   0.0      0.0      0.0      0.0      0.0   
            локационный           0.0      0.0      0.0      0.0      0.0   
            неподверженность      0.0      0.0      0.0      0.0      0.0   
            даякский              0.0      0.0      0.0      0.0      0.0   
            рэдкот                0.0      0.0      0.0      0.0      0.0   

                              topic_5  topic_6  topic_7  topic_8  topic_9  \
@lemmatized рлэ                   0.0      0.0      0.0      0.0      0.0   
            локационный           0.0      0.0      0.0      0.0      0.0   
            неподверженность      0.0      0.0      0.0      0.0      0.0   
            даякский              0.0      0.0      0.0      0.0      0.0   
            рэдкот                0.0      0.0      0.0      0.0      0.0   

                              ...  topic_12  topic_13  topic_14  topic_15  \
@lemmatized рлэ               ...       0.0       0.0       0.0       0.0   
            локационный       ...       0.0       0.0       0.0       0.0   
            неподверженность  ...       0.0       0.0       0.0       0.0   
            даякский          ...       0.0       0.0       0.0       0.0   
            рэдкот            ...       0.0       0.0       0.0       0.0   

                              topic_16  topic_17  topic_18  topic_19  \
@lemmatized рлэ                    0.0       0.0       0.0       0.0   
            локационный            0.0       0.0       0.0       0.0   
            неподверженность       0.0       0.0       0.0       0.0   
            даякский               0.0       0.0       0.0       0.0   
            рэдкот                 0.0       0.0       0.0       0.0   

                              topic_20  topic_21  
@lemmatized рлэ                    0.0       0.0  
            локационный            0.0       0.0  
            неподверженность       0.0       0.0  
            даякский               0.0       0.0  
            рэдкот                 0.0       0.0  

[5 rows x 22 columns]

In [91]:
bank_topics = optimizer._topic_bank.view_topics()

In [92]:
bank_topics.shape

(61688, 22)

In [93]:
bank_topics.head()

topic_0  topic_1  topic_2  topic_3  topic_4  \
@lemmatized рлэ                   0.0      0.0      0.0      0.0      0.0   
            локационный           0.0      0.0      0.0      0.0      0.0   
            неподверженность      0.0      0.0      0.0      0.0      0.0   
            даякский              0.0      0.0      0.0      0.0      0.0   
            рэдкот                0.0      0.0      0.0      0.0      0.0   

                              topic_5  topic_6  topic_7  topic_8  topic_9  \
@lemmatized рлэ                   0.0      0.0      0.0      0.0      0.0   
            локационный           0.0      0.0      0.0      0.0      0.0   
            неподверженность      0.0      0.0      0.0      0.0      0.0   
            даякский              0.0      0.0      0.0      0.0      0.0   
            рэдкот                0.0      0.0      0.0      0.0      0.0   

                              ...  topic_12  topic_13  topic_14  topic_15  \
@lemmatized рлэ               ...       0.0       0.0       0.0       0.0   
            локационный       ...       0.0       0.0       0.0       0.0   
            неподверженность  ...       0.0       0.0       0.0       0.0   
            даякский          ...       0.0       0.0       0.0       0.0   
            рэдкот            ...       0.0       0.0       0.0       0.0   

                              topic_16  topic_17  topic_18  topic_19  \
@lemmatized рлэ                    0.0       0.0       0.0       0.0   
            локационный            0.0       0.0       0.0       0.0   
            неподверженность       0.0       0.0       0.0       0.0   
            даякский               0.0       0.0       0.0       0.0   
            рэдкот                 0.0       0.0       0.0       0.0   

                              topic_20  topic_21  
@lemmatized рлэ                    0.0       0.0  
            локационный            0.0       0.0  
            неподверженность       0.0       0.0  
            даякский               0.0       0.0  
            рэдкот                 0.0       0.0  

[5 rows x 22 columns]

In [94]:
bank_topics['topic_5'].sort_values(ascending=False)[:20]

@lemmatized  игра           0.028238
             sonic          0.012376
             персонаж       0.010788
             версия         0.008670
             уровень        0.008087
             серия          0.007883
             сайт           0.005664
             игрок          0.005655
             герой          0.005283
             проект         0.004974
             final          0.004794
             fantasy        0.004590
             сюжет          0.004245
             мир            0.004234
             процесс        0.004171
             выпустить      0.004023
             playstation    0.003988
             хороший        0.003925
             игровой        0.003732
             hedgehog       0.003682
Name: topic_5, dtype: float64

And topic scores

In [95]:
optimizer._topic_bank.view_topic_scores()

,topic_0,topic_1,topic_2,topic_3,topic_4,topic_5,topic_6,topic_7,topic_8,topic_9,...,topic_12,topic_13,topic_14,topic_15,topic_16,topic_17,topic_18,topic_19,topic_20,topic_21
kernel_size,4995.000000,3361.000000,3652.000000,4251.000000,3710.000000,4818.000000,3604.000000,3997.000000,3164.000000,3199.000000,...,4409.000000,5424.000000,4725.000000,3860.000000,3918.000000,4646.000000,3871.000000,4977.000000,4024.000000,4207.000000
coherence_20,1.024162,1.195538,1.108464,1.062690,1.436570,1.007472,1.110294,1.001930,1.085643,1.817216,...,1.441053,1.090981,1.053423,1.082654,1.185210,1.015100,1.093561,1.230209,1.111099,1.092542
distance_to_nearest,0.000000,0.869286,0.874917,0.782039,0.725369,0.689765,0.858649,0.870273,0.889012,0.610435,...,0.730083,0.583112,0.736938,0.700414,0.855759,0.851559,0.544376,0.566511,0.641814,0.598573


All models are also saved (topics as $\Phi$ matrices and topic score values)

In [96]:
! ls $optimizer._topic_bank._path

model_0__phi.bin	    model_19__topic_scores.bin
model_0__topic_scores.bin   model_1__phi.bin
model_10__phi.bin	    model_1__topic_scores.bin
model_10__topic_scores.bin  model_2__phi.bin
model_11__phi.bin	    model_2__topic_scores.bin
model_11__topic_scores.bin  model_3__phi.bin
model_12__phi.bin	    model_3__topic_scores.bin
model_12__topic_scores.bin  model_4__phi.bin
model_13__phi.bin	    model_4__topic_scores.bin
model_13__topic_scores.bin  model_5__phi.bin
model_14__phi.bin	    model_5__topic_scores.bin
model_14__topic_scores.bin  model_6__phi.bin
model_15__phi.bin	    model_6__topic_scores.bin
model_15__topic_scores.bin  model_7__phi.bin
model_16__phi.bin	    model_7__topic_scores.bin
model_16__topic_scores.bin  model_8__phi.bin
model_17__phi.bin	    model_8__topic_scores.bin
model_17__topic_scores.bin  model_9__phi.bin
model_18__phi.bin	    model_9__topic_scores.bin
model_18__topic_scores.bin  topics.bin
model_19__phi.bin	    topic_scores.bin


In [109]:
optimizer._result.keys()

dict_keys(['optimum', 'optimum_std', 'bank_scores', 'bank_topic_scores', 'model_scores', 'model_topic_scores', 'num_bank_topics', 'num_model_topics'])

In [110]:
optimizer._result['num_bank_topics']

[5, 5, 6, 6, 6, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9]

In [111]:
len(optimizer._result['bank_topic_scores'])

20

In [112]:
optimizer._result

{'optimum': 9,
 'optimum_std': 0.0,
 'bank_scores': [{'perplexity_score': 20257.470703125,
   'coherence_20': 1.0863093667972832,
   'diversity_euclidean': 0.0619528730977368,
   'diversity_jensenshannon': 0.7010445276298223,
   'diversity_hellinger': 0.8168848068884893,
   'diversity_cosine': 0.8948451692109236,
   'perplexity': 20257.470703125,
   'ppl_fair': 20257.470703125,
   'ppl_cheatty': 7362.07666015625},
  {'perplexity_score': 20805.96484375,
   'coherence_20': 1.0628352670334091,
   'diversity_euclidean': 0.06146307847354089,
   'diversity_jensenshannon': 0.7009344857959255,
   'diversity_hellinger': 0.8172129050151774,
   'diversity_cosine': 0.8982419304818527,
   'perplexity': 20805.96484375,
   'ppl_fair': 20805.96484375,
   'ppl_cheatty': 7570.44873046875},
  {'perplexity_score': 18369.23046875,
   'coherence_20': 1.0340908704800558,
   'diversity_euclidean': 0.06003163324628971,
   'diversity_jensenshannon': 0.699532733844726,
   'diversity_hellinger': 0.814769049313561

In [113]:
optimizer._result['bank_topic_scores']

[[{'kernel_size': 5068,
   'coherence_20': 1.0297691355246918,
   'distance_to_nearest': 0.0},
  {'kernel_size': 5007,
   'coherence_20': 1.1756616093491055,
   'distance_to_nearest': 0.9197807889016681},
  {'kernel_size': 4070,
   'coherence_20': 1.3298810340740868,
   'distance_to_nearest': 0.8783019854111281},
  {'kernel_size': 4652,
   'coherence_20': 0.9104903714039343,
   'distance_to_nearest': 0.8806710301929691},
  {'kernel_size': 4803,
   'coherence_20': 0.9857446836345978,
   'distance_to_nearest': 0.8523585978250594}],
 [{'kernel_size': 5068,
   'coherence_20': 1.0297691355246918,
   'distance_to_nearest': 0.0},
  {'kernel_size': 5007,
   'coherence_20': 1.1756616093491055,
   'distance_to_nearest': 0.9197807889016681},
  {'kernel_size': 4070,
   'coherence_20': 1.3298810340740868,
   'distance_to_nearest': 0.8783019854111281},
  {'kernel_size': 4652,
   'coherence_20': 0.9104903714039343,
   'distance_to_nearest': 0.8806710301929691},
  {'kernel_size': 5384,
   'coherence_2

In [114]:
optimizer._result['model_scores'][0]

{'perplexity_score': 4860.8017578125,
 'coherence_20': 0.6750048319629702,
 'diversity_euclidean': 0.05157965876247023,
 'diversity_jensenshannon': 0.6647405644367678,
 'diversity_hellinger': 0.7666404062821004,
 'diversity_cosine': 0.8294436907285049,
 'perplexity': 4860.8017578125}

In [118]:
sum(s['coherence_20'] for s in optimizer._result['bank_topic_scores'][-1]) / 9

1.0198770555312775

In [119]:
len(optimizer._result['bank_scores'])

20

In [120]:
optimizer._result['bank_scores'][-1]

{'perplexity_score': 12956.1708984375,
 'coherence_20': 1.0198770555312777,
 'diversity_euclidean': 0.0630011660210022,
 'diversity_jensenshannon': 0.6894934345906027,
 'diversity_hellinger': 0.8011505487381404,
 'diversity_cosine': 0.8798738518529864,
 'perplexity': 12956.1708984375,
 'ppl_fair': 12956.1708984375,
 'ppl_cheatty': 6915.63525390625}

In [121]:
optimizer._result['model_topic_scores']

[[{'kernel_size': 5842, 'coherence_20': 0.6362738850913598},
  {'kernel_size': 5796, 'coherence_20': 0.7017274275328904},
  {'kernel_size': 4873, 'coherence_20': 0.4808373246649988},
  {'kernel_size': 4946, 'coherence_20': 0.5641984460697425},
  {'kernel_size': 5715, 'coherence_20': 0.5445732395217968},
  {'kernel_size': 5068, 'coherence_20': 0.4520213308964054},
  {'kernel_size': 5068,
   'coherence_20': 1.0297691355246918,
   'distance_to_nearest': 0.0},
  {'kernel_size': 5805, 'coherence_20': 0.5174446740493418},
  {'kernel_size': 5254, 'coherence_20': 0.4274255286876996},
  {'kernel_size': 5007,
   'coherence_20': 1.1756616093491055,
   'distance_to_nearest': 0.9197807889016681},
  {'kernel_size': 5578, 'coherence_20': 0.6489807489743467},
  {'kernel_size': 4070,
   'coherence_20': 1.3298810340740868,
   'distance_to_nearest': 0.8783019854111281},
  {'kernel_size': 4652,
   'coherence_20': 0.9104903714039343,
   'distance_to_nearest': 0.8806710301929691},
  {'kernel_size': 4433, 'c

In [123]:
! echo $SEARCH_RESULTS_FOLDER_PATH
! ls $SEARCH_RESULTS_FOLDER_PATH

./Good_RU_Wiki__internals/result2
bank__0  search_result__0.json


In [124]:
! ls Good_RU_Wiki__internals

batches  dict.dict  result  result2  vw.txt
